# 05 · IDUN raw → LSL alignment

Align native IDUN EEG content to cap EEG on the LSL clock. This notebook saves only mapping and QC detail; step 10 writes the compact per-sample files and derives ET raw time through step 01 when available.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (PROJECT_ROOT / 'helpers').is_dir():
        sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise FileNotFoundError('Could not locate the BIGPROJECT/helpers folder')

from helpers.project_config import load_project_config
from helpers.sync_quality import idun_lsl_alignment as alignment
importlib.reload(alignment)

CONFIG = load_project_config(PROJECT_ROOT / 'ImBrAid' / 'config.json')
REPAIR_ROOT = CONFIG.path('paths', 'idun_repair_root')
ALIGNMENT_CSV = CONFIG.path('files', 'idun_lsl_alignment_csv')
CHANNELS_CSV = CONFIG.path('files', 'idun_lsl_channels_csv')
WINDOWS_CSV = CONFIG.path('files', 'idun_lsl_windows_csv')
ISSUES_CSV = CONFIG.path('files', 'idun_lsl_issues_csv')
OVERWRITE = True
PARTICIPANTS = None
VISITS = None
CAP_STREAM_NAME = CONFIG.value('modalities', 'cap_eeg_stream_name')
BASE_HZ = 50.0
FEATURE_HZ = 10.0
WINDOW_S = 300.0
STEP_S = 150.0
COARSE_SEARCH_RADIUS_S = 30.0
LOCAL_SEARCH_RADIUS_S = 2.0
MINIMUM_PEAK_CORRELATION = 0.35
MINIMUM_PEAK_MARGIN = 0.02
REVIEW_DRIFT_PPM = 2000.0
MAXIMUM_DRIFT_PPM = 10000.0

In [2]:
jobs = alignment.discover_cap_eeg_sync_jobs(
    REPAIR_ROOT, participants=PARTICIPANTS, visits=VISITS,
)
display(jobs[['pid', 'visit', 'discovery_status']])

,pid,visit,discovery_status
0,A7G5SM,VX,idun_repair_unusable
1,ABVLGQ,VX,ready
2,B9NYQ8,VX,idun_repair_unusable
3,CRVWF9,VX,ready
4,CRYSEP,VX,ready
5,CU6DM4,VX,ready
6,DUJL2L,VX,ready
7,E4MKXT,VX,ready
8,EASFXM,VX,ready
9,FEDZAG,VX,ready


In [3]:
mappings, channels, windows, issues = alignment.build_idun_cap_alignments(
    jobs, cap_stream_name=CAP_STREAM_NAME, base_hz=BASE_HZ,
    feature_hz=FEATURE_HZ, window_s=WINDOW_S, step_s=STEP_S,
    coarse_search_radius_s=COARSE_SEARCH_RADIUS_S,
    local_search_radius_s=LOCAL_SEARCH_RADIUS_S,
    minimum_peak_correlation=MINIMUM_PEAK_CORRELATION,
    minimum_peak_margin=MINIMUM_PEAK_MARGIN,
    review_drift_ppm=REVIEW_DRIFT_PPM,
    maximum_drift_ppm=MAXIMUM_DRIFT_PPM,
)
if OVERWRITE:
    alignment.save_idun_cap_alignment_results(
        mappings, channels, windows, issues, alignment_csv=ALIGNMENT_CSV,
        channels_csv=CHANNELS_CSV, windows_csv=WINDOWS_CSV,
        issues_csv=ISSUES_CSV,
    )
display(mappings[[
    'pid', 'visit', 'mapping_status', 'usable', 'quality_grade',
    'review_required', 'selected_cap_feature', 'drift_ppm',
    'n_good_windows', 'fit_residual_p95_s', 'channel_agreement',
    'mapping_reason', 'warnings',
]])
display(issues if not issues.empty else pd.DataFrame({'status': ['no issues']}))

,pid,visit,mapping_status,usable,quality_grade,review_required,selected_cap_feature,drift_ppm,n_good_windows,fit_residual_p95_s,channel_agreement,mapping_reason,warnings
0,A7G5SM,VX,unavailable,False,D,True,NaN,NaN,NaN,NaN,NaN,Discovery status: idun_repair_unusable,Discovery status: idun_repair_unusable
1,ABVLGQ,VX,affine,True,A,False,T7-T8,-569.032235,17.0,0.018729,True,Supported affine mapping from IDUN and cap EEG...,
2,B9NYQ8,VX,unavailable,False,D,True,NaN,NaN,NaN,NaN,NaN,Discovery status: idun_repair_unusable,Discovery status: idun_repair_unusable
3,CRVWF9,VX,affine,True,A,False,T7-T8,-1107.618332,16.0,0.056666,True,Supported affine mapping from IDUN and cap EEG...,
4,CRYSEP,VX,affine,True,A,False,T7-T8,-345.416163,6.0,0.028706,True,Supported affine mapping from IDUN and cap EEG...,
5,CU6DM4,VX,affine,True,A,False,T7-T8,-428.468097,17.0,0.035443,True,Supported affine mapping from IDUN and cap EEG...,
6,DUJL2L,VX,affine_review,True,B,True,T8-M1,-762.212185,17.0,0.071696,True,Usable affine mapping with review-level qualit...,median peak correlation was 0.593
7,E4MKXT,VX,affine,True,A,False,T7-T8,-836.290677,16.0,0.047424,True,Supported affine mapping from IDUN and cap EEG...,
8,EASFXM,VX,affine,True,A,False,T7-T8,-1006.836460,17.0,0.054746,True,Supported affine mapping from IDUN and cap EEG...,
9,FEDZAG,VX,affine,True,A,False,T7-T8,-857.608382,16.0,0.052702,True,Supported affine mapping from IDUN and cap EEG...,


,project,pid,visit,severity,issue,detail
0,ImBrAid,A7G5SM,VX,error,idun_cap_source_unavailable,Discovery status: idun_repair_unusable
1,ImBrAid,B9NYQ8,VX,error,idun_cap_source_unavailable,Discovery status: idun_repair_unusable
2,ImBrAid,DUJL2L,VX,warning,idun_cap_mapping_review,median peak correlation was 0.593
3,ImBrAid,FGJK2B,VX,error,idun_cap_source_unavailable,Discovery status: idun_repair_unusable
4,ImBrAid,HKJPM3,VX,error,idun_cap_source_unavailable,Discovery status: idun_repair_unusable
5,ImBrAid,HNWSPB,VX,error,idun_cap_source_unavailable,Discovery status: idun_repair_unusable
6,ImBrAid,JVC62S,VX,error,idun_cap_source_unavailable,Discovery status: idun_repair_unusable
7,ImBrAid,N37VFK,VX,error,idun_cap_source_unavailable,Discovery status: idun_repair_unusable
8,ImBrAid,N7C4A6,VX,error,idun_cap_source_unavailable,Discovery status: idun_repair_unusable
9,ImBrAid,NJN65T,VX,error,idun_cap_source_unavailable,Discovery status: idun_repair_unusable
